# dim_gender.py

## Purpose
Builds `dim_gender` as an SCD1 dimension with the EA FC gender categories.

## Source
- Bronze: `fifa.bronze.ea_fc_players_raw` (column `gender`)

## Strategy
- SCD1 (categories are stable)
- Overwrite mode
- Hardcoded mapping to guarantee ID stability:
  - `1` = Men's Football
  - `2` = Women's Football
  - `99` = unknown / fallback

## Output
- Table: `fifa.silver.dim_gender`

## Columns
- `gender_id` (INT)
- `gender_name` (STRING)
- `_ingestion_timestamp` (TIMESTAMP)

## Notes
The hardcoded mapping ensures IDs never change, even if new genders are added in the future. New values would fall into the `99` fallback until explicitly mapped.

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit, when

# ===== Read Bronze =====
bronze = spark.read.table("fifa.bronze.ea_fc_players_raw")

# Build the dimension
dim_gender = (
    bronze
    .select("gender")
    .distinct()
    # Hardcoded mapping: ensures IDs never change
    .withColumn("gender_id",
        when(col("gender") == "Men's Football", lit(1))
        .when(col("gender") == "Women's Football", lit(2))
        .otherwise(lit(99))  # fallback for unknown values
    )
    .withColumn("gender_name", col("gender"))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .select("gender_id", "gender_name", "_ingestion_timestamp")
    .orderBy("gender_id")
)

# ===== Create Silver table =====
spark.sql("""
CREATE TABLE IF NOT EXISTS fifa.silver.dim_gender (
    gender_id INT,
    gender_name STRING,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA;
""")

# Overwrite the table (small catalog)
(dim_gender
 .write
 .mode("overwrite")
 .option("mergeSchema", "true")
 .saveAsTable("fifa.silver.dim_gender")
)

print(f"dim_gender created with {spark.read.table('fifa.silver.dim_gender').count()} rows")

LIMIT 10 of the dim_gender table. Quick verification that we have entered the raw data into our table.

In [0]:
%sql
SELECT * FROM fifa.silver.dim_gender ORDER BY gender_id limit 10;